In [1]:
import time
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import itertools

wd='PATH_TO_ENTTEMPLATES_DATA_ROOT/Analysis/python_Similarity/data/'
output='PATH_TO_ENTTEMPLATES_DATA_ROOT/Analysis/python_Similarity/output/'

# Load pre-trained SentenceTransformer
sbert_model = SentenceTransformer('all-distilroberta-v1')

# Load data
df = pd.read_pickle(wd+'company_embedding_ST')
df = df.sort_values(by=['fullname'])
df=df[df['fullname']== 'AI ML|Vertical Applications|Consumer']

In [2]:
df

,companyid,fullname,embedding
254864,56133-10,AI ML|Vertical Applications|Consumer,"[-7.123363e-05, -0.033342585, 0.05527848, -0.0..."
328339,439013-71,AI ML|Vertical Applications|Consumer,"[-0.016678385, -0.034818947, -0.008875517, 0.0..."
39636,399714-85,AI ML|Vertical Applications|Consumer,"[-0.0065959916, -0.050884463, -0.006652736, -0..."
254653,53503-48,AI ML|Vertical Applications|Consumer,"[-0.04566997, -0.043910053, 0.0076927086, -0.0..."
206602,160346-89,AI ML|Vertical Applications|Consumer,"[-0.025326159, -0.036783617, 0.014422003, -0.0..."
...,...,...,...
327977,110776-69,AI ML|Vertical Applications|Consumer,"[-0.025744675, -0.06513593, -0.009325135, -0.0..."
37862,315972-73,AI ML|Vertical Applications|Consumer,"[-0.002767625, -0.027123047, 0.02105413, 0.005..."
210634,462453-13,AI ML|Vertical Applications|Consumer,"[-0.034871046, -0.021072688, -1.1106268e-05, -..."
155898,83752-75,AI ML|Vertical Applications|Consumer,"[-0.03193334, 0.002922773, 0.04413781, -0.0466..."


In [11]:
names_to_exclude = [
    'AI ML|AI & ML Semiconductors|Edge AI Software',
    'AI ML|AI & ML Semiconductors|Intelligent Sensors & Devices',
    'AI ML|AI & ML Semiconductors|Processor Design',
    'AI ML|Autonomous Machines|Autonomous Vehicles',
    'AI ML|Autonomous Machines|Intelligent Robotics',
    'AI ML|Horizontal Platforms|AI Automation Platforms',
    'AI ML|Horizontal Platforms|AI Core',
    'AI ML|Horizontal Platforms|Computer Vision',
    'AI ML|Horizontal Platforms|Natural Language Technology',
    'AI ML|Vertical Applications|Consumer',
    'AI ML|Vertical Applications|Financial Services',
    'AI ML|Vertical Applications|Healthcare',
    "AI ML|Vertical Applications|Industrial",
    "AI ML|Vertical Applications|Information Technology",
    "AI ML|Vertical Applications|Mobility",
    "AgTech|Ag biotech|Animal biotech",
    "AgTech|Ag biotech|Biomaterials",
    "AgTech|Ag biotech|Plant biotech",
    "AgTech|Ag biotech|Plant data & analysis",
    "AgTech|Agrifinance & eCommerce|Agribusiness marketplaces",
    "AgTech|Agrifinance & eCommerce|Finance & insurance",
    "AgTech|Animal ag|Aquaculture",
    "AgTech|Animal ag|Insect farming",
    "AgTech|Animal ag|Livestock & land animal tech",
    "AgTech|Animal ag|Pollination tech",
    "AgTech|Indoor farming|Components",
    "AgTech|Indoor farming|Growers",
    "AgTech|Indoor farming|Systems",
    "AgTech|Precision ag|Drones & imagery analytics",
    "AgTech|Precision ag|Farm management software",
    "AgTech|Precision ag|Field IoT",
    "AgTech|Precision ag|Robotics & smart field equipment",
    "Blockchain|Blockchain|Blockchain",
    "Blockchain|Enterprise|Enterprise",
    "Blockchain|Financial Services|Financial Services",
    "Blockchain|Infrastructure|Infrastructure",
    "Blockchain|NFTs|NFTs",
    "Carbon and Emissions Tech|Built Environment|Building Energy Efficiency",
    "Carbon and Emissions Tech|Built Environment|Green Construction",
    "Carbon and Emissions Tech|Built Environment|Heating and Cooling",
    "Carbon and Emissions Tech|Carbon Tech|Biological Carbon Removal",
    "Carbon and Emissions Tech|Carbon Tech|Carbon Accounting/Analytics",
]


df = df[~df['fullname'].isin(names_to_exclude)]
df

,companyid,fullname,embedding
13679,54707-41,Carbon and Emissions Tech|Carbon Tech|Carbon F...,"[-0.021155236, -0.002549114, -0.0106962565, -0..."
271785,115456-06,Carbon and Emissions Tech|Carbon Tech|Carbon F...,"[0.0025053257, 0.013508373, -0.0143952, -0.030..."
224232,53713-81,Carbon and Emissions Tech|Carbon Tech|Carbon F...,"[-0.00012465817, 0.0038097452, 0.016205737, -0..."
238794,53755-30,Carbon and Emissions Tech|Carbon Tech|Carbon F...,"[0.01175531, 0.034781028, -0.023994312, -0.052..."
119779,484154-74,Carbon and Emissions Tech|Carbon Tech|Carbon F...,"[-0.017856784, -0.002753013, -0.009651142, -0...."
...,...,...,...
284241,110508-31,Supply Chain Tech|Warehousing tech|Warehouse a...,"[-0.020348571, -0.007767355, -0.023641346, 0.0..."
39323,232395-04,Supply Chain Tech|Warehousing tech|Warehouse a...,"[-0.011821735, -0.028306404, -0.0102774585, 0...."
88496,148163-68,Supply Chain Tech|Warehousing tech|Warehouse a...,"[-0.029778404, -0.020195313, 0.020117922, 0.01..."
59489,251407-00,Supply Chain Tech|Warehousing tech|Warehouse a...,"[-0.048782956, -0.042804226, 0.01670886, 0.026..."


In [3]:
# Initialize list to store results
results = []

# Initialize counter and timer
counter = 0
start_time = time.time()

# Group by 'fullname' and iterate over each group
for name, group in df.groupby('fullname'):
    # Get all company pairs within this group
    print('working on '+name)
    for (idx1, row1), (idx2, row2) in itertools.combinations(group.iterrows(), 2):
        # Start timer for this calculation
        calc_start_time = time.time()
        # Calculate similarity between the two companies' descriptions
        similarity = cosine_similarity([row1['embedding']], [row2['embedding']])[0][0]
        
        # Store the result
        results.append({
            'companyid1': row1['companyid'],
            'fullname1': row1['fullname'],
            'companyid2': row2['companyid'],
            'fullname2': row2['fullname'],
            'similarity': similarity
        })
        
        # Increment counter and check if it's a multiple of 1000
        counter += 1
        if counter % 1000000 == 0:
            # Calculate time consumed
            total_time = time.time() - start_time
            print(f'{counter/1000000} million finished; time consumed in total: {total_time:.2f} seconds.')
    similarity_df_ST = pd.DataFrame(results)
    name = name.replace("/", "_")
    similarity_df_ST.to_csv(output+'similarity_ST_'+ name + '.csv')
    print(name+' finished')
    results = []

working on AI ML|Vertical Applications|Consumer
1.0 million finished; time consumed in total: 74.58 seconds.
2.0 million finished; time consumed in total: 148.27 seconds.
3.0 million finished; time consumed in total: 221.27 seconds.
4.0 million finished; time consumed in total: 293.68 seconds.
5.0 million finished; time consumed in total: 366.44 seconds.
6.0 million finished; time consumed in total: 439.39 seconds.
7.0 million finished; time consumed in total: 511.94 seconds.
8.0 million finished; time consumed in total: 584.99 seconds.
9.0 million finished; time consumed in total: 658.46 seconds.
10.0 million finished; time consumed in total: 732.04 seconds.
11.0 million finished; time consumed in total: 805.79 seconds.
12.0 million finished; time consumed in total: 878.43 seconds.
13.0 million finished; time consumed in total: 952.82 seconds.
14.0 million finished; time consumed in total: 1024.50 seconds.
15.0 million finished; time consumed in total: 1097.83 seconds.
16.0 million fi

In [6]:
similarity_df_ST

,companyid1,fullname1,companyid2,fullname2,similarity
0,463332-34,Carbon and Emissions Tech|Carbon Tech|Carbon A...,459260-29,Carbon and Emissions Tech|Carbon Tech|Carbon A...,0.749403
1,463332-34,Carbon and Emissions Tech|Carbon Tech|Carbon A...,92083-06,Carbon and Emissions Tech|Carbon Tech|Carbon A...,0.727068
2,463332-34,Carbon and Emissions Tech|Carbon Tech|Carbon A...,85885-57,Carbon and Emissions Tech|Carbon Tech|Carbon A...,0.680855
3,463332-34,Carbon and Emissions Tech|Carbon Tech|Carbon A...,416571-04,Carbon and Emissions Tech|Carbon Tech|Carbon A...,0.860592
4,463332-34,Carbon and Emissions Tech|Carbon Tech|Carbon A...,442986-67,Carbon and Emissions Tech|Carbon Tech|Carbon A...,0.770729
...,...,...,...,...,...
11170,232655-77,Carbon and Emissions Tech|Carbon Tech|Carbon A...,437142-07,Carbon and Emissions Tech|Carbon Tech|Carbon A...,0.530019
11171,232655-77,Carbon and Emissions Tech|Carbon Tech|Carbon A...,279496-27,Carbon and Emissions Tech|Carbon Tech|Carbon A...,0.587416
11172,343107-46,Carbon and Emissions Tech|Carbon Tech|Carbon A...,437142-07,Carbon and Emissions Tech|Carbon Tech|Carbon A...,0.738416
11173,343107-46,Carbon and Emissions Tech|Carbon Tech|Carbon A...,279496-27,Carbon and Emissions Tech|Carbon Tech|Carbon A...,0.500635
